# Challenge W4: NLP - Fake News


Antonio Traquinas
## MODEL_AT_5 - Transfer Learning: Fine-tuned BERT (bert-base-uncased)


<br>

<hr>

## Steps:
    1. Import data
    2. EDA
    3. Data Splitting
    4. Tokenization (BERT tokenizer)
    5. Dataset / DataLoader preparation
    6. Model fine-tuning (BERT)
    7. Evaluation
    8. Save model

##  Libraries:

In [2]:
# pip install transformers datasets evaluate accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
#import seaborn as sns

import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
from torch.utils.data import Dataset

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# 1. Load data

In [5]:
df = pd.read_csv("./dataset/data.csv", engine='python', on_bad_lines='warn')

In [6]:
df.shape
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39942 entries, 0 to 39941
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    39942 non-null  int64 
 1   title    39942 non-null  object
 2   text     39942 non-null  object
 3   subject  39942 non-null  object
 4   date     39942 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.5+ MB


# 2. EDA

In [7]:
df.isnull().sum() # missing values?

,0
label,0
title,0
text,0
subject,0
date,0


In [8]:
print("Duplicate rows:", df.duplicated().sum()) # Duplicates?


Duplicate rows: 201


In [9]:
df.sample(5, random_state=42)

,label,title,text,subject,date
6524,1,Oil business seen in strong position as Trump ...,(This January 3 story was corrected to remove...,politicsNews,"January 3, 2017"
30902,0,WHOA! COLLEGE SNOWFLAKE FREAKS OUT: Screams Fo...,So much for healthy debate on college campus I...,politics,"May 12, 2017"
36459,0,CRONY CORRUPT POLITICS: Obama Admin BLOCKED FB...,The information is spilling out little by litt...,Government News,"Aug 11, 2016"
9801,1,Cruz campaign vetting Fiorina as a possible VP...,WASHINGTON (Reuters) - U.S. Republican preside...,politicsNews,"April 25, 2016"
25638,0,Minnesota Woman Writes Amazing F*ck Off Lette...,"Attention, conservative men. This one is for y...",News,"July 2, 2016"


In [10]:
print("Unique subjects:", df['subject'].nunique())
df['subject'].value_counts()

Unique subjects: 6


,count
subject,
politicsNews,11272
News,9050
worldnews,8727
politics,6841
left-news,2482
Government News,1570


In [11]:
pd.crosstab(df['subject'], df['label']) # Because subject and label are correlated we should drop "subject" column to avoid data leakage.

label,0,1
subject,,
Government News,1570,0
News,9050,0
left-news,2482,0
politics,6841,0
politicsNews,0,11272
worldnews,0,8727


# 3. TRAIN TEST VALIDATION SPLIT

In [12]:
# Dropping columns subject and date as the have been shown to not carry important information
df = df.drop(columns=['subject','date'])

In [13]:
df = df.drop_duplicates() # Dropping 201 duplicates that appeared once subject and date were removed (meaning that they had same title and/or text and we
                            #could fall into data leakage)
print(df.shape)

(36429, 3)


In [14]:

# Split into features and target
# Adjust the target column name to match your dataset (e.g. 'label', 'class', 'Spam/Ham')
X = df['title'] + ' ' + df['text']   # <-- replace 'label' with your actual target column
y = df['label']

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: Split the remaining 30% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Display dataset shapes
print(f"Train shape:      {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape:       {X_test.shape}")

# Display class balance
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True))

print("\nValidation class balance:")
print(y_val.value_counts(normalize=True))

print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

Train shape:      (25500,)
Validation shape: (5464,)
Test shape:       (5465,)

Train class balance:
label
1    0.543216
0    0.456784
Name: proportion, dtype: float64

Validation class balance:
label
1    0.543192
0    0.456808
Name: proportion, dtype: float64

Test class balance:
label
1    0.543275
0    0.456725
Name: proportion, dtype: float64


# 4. Tokenization (BERT)
    - Unlike the classical ML pipeline, BERT does not need manual lowercasing, stopword removal,
      or lemmatization: the WordPiece tokenizer and the pretrained model already encode this kind
      of information. We only do light cleanup (whitespace/URL noise) and let the tokenizer handle
      the rest.

In [15]:
def light_clean(text):
    text = re.sub(r'http\S+|www\.\S+', ' ', text)   # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()          # collapse whitespace
    return text

X_train_clean = X_train.apply(light_clean)
X_val_clean   = X_val.apply(light_clean)
X_test_clean  = X_test.apply(light_clean)

In [16]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256  # truncate/pad sequences to this many tokens

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_batch(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )

train_encodings = tokenize_batch(X_train_clean)
val_encodings   = tokenize_batch(X_val_clean)
test_encodings  = tokenize_batch(X_test_clean)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# 5. Dataset / DataLoader preparation
    - Wrap the tokenized tensors + labels into a torch Dataset that the HuggingFace Trainer can consume.

In [17]:
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = FakeNewsDataset(train_encodings, y_train)
val_dataset   = FakeNewsDataset(val_encodings, y_val)
test_dataset  = FakeNewsDataset(test_encodings, y_test)

print(f"Train dataset size:      {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size:       {len(test_dataset)}")

Train dataset size:      25500
Validation dataset size: 5464
Test dataset size:       5465


# 6. Model fine-tuning - BERT
    - Load bert-base-uncased with a binary classification head and fine-tune it end-to-end
      on the fake news training set using HuggingFace's Trainer API.

In [18]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [19]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

In [20]:
training_args = TrainingArguments(
    output_dir="./bert_fake_news_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),  # mixed precision if a GPU is available
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [21]:
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.008378,0.009594,0.998170,0.998170
2,0.004559,0.001025,0.999634,0.999634
3,0.000013,0.000253,0.999817,0.999817


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=4782, training_loss=0.016899974713155685, metrics={'train_runtime': 1312.4497, 'train_samples_per_second': 58.288, 'train_steps_per_second': 3.644, 'total_flos': 1.006399786752e+16, 'train_loss': 0.016899974713155685, 'epoch': 3.0})


# 7. Evaluation
    - Evaluate the fine-tuned model on the validation set, then confirm performance on the
      held-out test set.

In [22]:
val_pred_output = trainer.predict(val_dataset)
y_val_pred = np.argmax(val_pred_output.predictions, axis=-1)

print("=== Fine-tuned BERT - Validation ===")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))
print(confusion_matrix(y_val, y_val_pred))

=== Fine-tuned BERT - Validation ===
Accuracy: 0.9998169838945827
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2496
           1       1.00      1.00      1.00      2968

    accuracy                           1.00      5464
   macro avg       1.00      1.00      1.00      5464
weighted avg       1.00      1.00      1.00      5464

[[2496    0]
 [   1 2967]]


In [23]:
test_pred_output = trainer.predict(test_dataset)
y_test_pred = np.argmax(test_pred_output.predictions, axis=-1)

print("=== Fine-tuned BERT - Test ===")
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))

=== Fine-tuned BERT - Test ===
Test Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2496
           1       1.00      1.00      1.00      2969

    accuracy                           1.00      5465
   macro avg       1.00      1.00      1.00      5465
weighted avg       1.00      1.00      1.00      5465

[[2496    0]
 [   0 2969]]


# 8. Save model
    - Persist the fine-tuned model and tokenizer so they can be reloaded later with
      `BertForSequenceClassification.from_pretrained(...)` / `BertTokenizerFast.from_pretrained(...)`.

In [24]:
SAVE_DIR = "./bert_fake_news_finetuned"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model and tokenizer saved to {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./bert_fake_news_finetuned


### Notes
- This fine-tunes the full `bert-base-uncased` model (~110M parameters). A GPU is strongly
  recommended; on CPU only, reduce `MAX_LEN`, batch size, and/or use a smaller model such as
  `distilbert-base-uncased` for faster iteration.
- Install requirements if needed: `pip install transformers datasets evaluate accelerate torch`.
- To try a lighter/faster transfer-learning baseline, swap `MODEL_NAME` for `"distilbert-base-uncased"`
  (and use `DistilBertTokenizerFast` / `DistilBertForSequenceClassification`).